# Benchmark: Classical ML Models — Auto-Loop

**Models** : Linear Regression · SVR · XGBoost · MLP  
**Author** : Anwesha Singh — CSE, Manipal University Jaipur

---

## How to use

| Cell | Action |
|---|---|
| **Cell 1** | Run once per kernel session |
| **Cell 2** | Set `HORIZONS_TO_RUN` — then never touch again |
| **Cell 3** | Run and walk away — all horizons evaluated automatically |
| **Cell 4** | Cross-horizon summary table |

Loads **frozen canonical splits** produced by `01_NiOA_DRNN_Training.ipynb`.  
Do **not** regenerate splits here — identical data must be used for all models.

### v2 changes
- `log_transform=TARGET_LOG_TRANSFORM` added to all `save_benchmark_results` calls  
  so that metrics are reported in original kWh units rather than log1p space.
- RAM cap (`MAX_ML_ROWS = 100_000`) added for LR and MLP — the full flattened  
  training array is ~18.5 GB and would cause OOM on most systems.
- Models saved via `joblib` for full reproducibility.
- Scatter / residual plots produced in original kWh space (after expm1 inversion).
- `run_config.json` saved per horizon for audit trail.


In [1]:
!pip install xgboost

In [2]:
import os, sys, json, gc, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from sklearn.linear_model    import LinearRegression
from sklearn.svm             import SVR
from sklearn.neural_network  import MLPRegressor
from xgboost                 import XGBRegressor

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
for _p in [PROJECT_ROOT, os.path.join(PROJECT_ROOT, 'core')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from core.config  import (
    SPLITS_DIR, BENCHMARK_DIR, RESULTS_DIR,
    FORECAST_HORIZONS, RANDOM_SEED, TARGET_LOG_TRANSFORM,
)
from benchmarking.utils.data_loader import load_splits, flatten_sequences, combine_train_val
from benchmarking.utils.metrics     import compute_metrics, save_benchmark_results, build_comparison_table

sns.set(style='whitegrid')
ANALYSIS_DIR = os.path.join(RESULTS_DIR, 'analysis')
os.makedirs(ANALYSIS_DIR, exist_ok=True)
print(f'Project root      : {PROJECT_ROOT}')
print(f'log_transform     : {TARGET_LOG_TRANSFORM}')
print('Environment ready.')

Project root      : D:\pipeline5_2
log_transform     : True
Environment ready.


In [3]:
# ── Edit these lines only ─────────────────────────────────────────────────────
HORIZONS_TO_RUN = [60, 300, 900, 1800]
SKIP_COMPLETED  = False   # True = skip horizons that already have saved results
# ─────────────────────────────────────────────────────────────────────────────

# SVR max training rows (SVR is O(n²) — cap to keep runtime tractable)
MAX_SVR_ROWS = 50_000

# LR / MLP row cap — the flattened X_tv_flat is (2.27M × 2040) ≈ 18.5 GB.
# A random subset of 100k rows is used for LR and MLP training.
# XGBoost handles large data natively via its tree partitioning; it uses full splits.
MAX_ML_ROWS = 100_000

for _h in HORIZONS_TO_RUN:
    assert _h in FORECAST_HORIZONS, f'{_h} not in {FORECAST_HORIZONS}'

print(f'Horizons queued : {HORIZONS_TO_RUN}')
print(f'Skip completed  : {SKIP_COMPLETED}')
print(f'SVR row cap     : {MAX_SVR_ROWS:,}')
print(f'LR/MLP row cap  : {MAX_ML_ROWS:,}')

Horizons queued : [60, 300, 900, 1800]
Skip completed  : False
SVR row cap     : 50,000
LR/MLP row cap  : 100,000


---
## Main Loop — all four classical ML models per horizon

In [4]:
all_results = {}   # {horizon: {model: metrics_dict}}
loop_start  = time.time()

for _loop_idx, HORIZON in enumerate(HORIZONS_TO_RUN):

    _elapsed = (time.time() - loop_start) / 3600
    print(f'\n' + '=' * 60)
    print(f'  Horizon {_loop_idx+1}/{len(HORIZONS_TO_RUN)} : '
          f'k = {HORIZON} s   {_elapsed:.2f} h elapsed')
    print('=' * 60)

    # ── Skip check ──────────────────────────────────────────────────────────
    if SKIP_COMPLETED:
        _csv_path = os.path.join(BENCHMARK_DIR, f'horizon_{HORIZON}', 'summary_classical.csv')
        if os.path.isfile(_csv_path):
            print(f'  [SKIP] summary already exists — {_csv_path}')
            continue

    # ── Create output directories ────────────────────────────────────────────
    _horizon_dir = os.path.join(BENCHMARK_DIR, f'horizon_{HORIZON}')
    _plots_dir   = os.path.join(_horizon_dir, 'classical_plots')
    _models_dir  = os.path.join(_horizon_dir, 'classical_models')
    for _d in [_horizon_dir, _plots_dir, _models_dir]:
        os.makedirs(_d, exist_ok=True)

    # ── Load splits ─────────────────────────────────────────────────────────
    print(f'  Loading canonical splits ...')
    X_train, y_train, X_val, y_val, X_test, y_test, scaler, meta = \
        load_splits(SPLITS_DIR, HORIZON)

    # Flatten: (N, seq_len, feats) → (N, seq_len × feats)
    X_tr_flat = flatten_sequences(X_train)
    X_va_flat = flatten_sequences(X_val)
    X_te_flat = flatten_sequences(X_test)

    # Combined train + val for models without built-in early stopping
    X_tv_flat, y_tv = combine_train_val(X_tr_flat, y_train, X_va_flat, y_val)

    print(f'  Flat train+val : {X_tv_flat.shape}  |  test : {X_te_flat.shape}')
    print(f'  Approx RAM (full TV flat) : {X_tv_flat.nbytes / 1e9:.2f} GB')

    # ── RAM cap for LR / MLP — random stratified subset ─────────────────────
    # The flattened array at full size is ~18.5 GB and causes OOM on most
    # systems.  A reproducible random sample of MAX_ML_ROWS rows is used
    # for LR and MLP.  XGBoost uses the full X_tr_flat / X_va_flat directly.
    if len(X_tv_flat) > MAX_ML_ROWS:
        _rng = np.random.RandomState(RANDOM_SEED)
        _idx = _rng.choice(len(X_tv_flat), MAX_ML_ROWS, replace=False)
        _idx.sort()   # keep temporal order within the sample
        X_tv_cap = X_tv_flat[_idx]
        y_tv_cap  = y_tv[_idx]
        print(f'  LR/MLP capped to {MAX_ML_ROWS:,} rows  '
              f'(from {len(X_tv_flat):,})')
    else:
        X_tv_cap = X_tv_flat
        y_tv_cap  = y_tv

    # ── Helper: save scatter + residual plots in kWh space ──────────────────
    def _save_pred_plot(y_true_log1p, y_pred_log1p, model_name):
        """
        Both inputs are in log1p space (matching the canonical splits).
        Plots are produced in original kWh units after expm1 inversion
        so they are directly comparable to the NiOA-DRNN result plots.
        """
        if TARGET_LOG_TRANSFORM:
            _yt = np.expm1(np.clip(y_true_log1p.astype(np.float64), -10., 30.))
            _yp = np.maximum(
                np.expm1(np.clip(y_pred_log1p.astype(np.float64), -10., 30.)), 0.
            )
        else:
            _yt = y_true_log1p
            _yp = np.maximum(y_pred_log1p, 0.)

        # Scatter
        _fig, _ax = plt.subplots(figsize=(5, 5))
        _ax.scatter(_yt, _yp, alpha=0.3, s=4, color='steelblue')
        _lim = [min(_yt.min(), _yp.min()), max(_yt.max(), _yp.max())]
        _ax.plot(_lim, _lim, 'r--', lw=1.5, label='Perfect')
        _ax.set_xlabel(f'Actual ΔE  k={HORIZON}s (kWh)')
        _ax.set_ylabel(f'Predicted ΔE  k={HORIZON}s (kWh)')
        _ax.set_title(f'{model_name} — Predicted vs Actual  k={HORIZON}s')
        _ax.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(_plots_dir, f'{model_name}_scatter.png'), dpi=180)
        plt.close('all')

        # Residuals
        _fig, _ax = plt.subplots(figsize=(7, 3))
        sns.histplot(_yt - _yp, bins=50, kde=True, ax=_ax, color='steelblue')
        _ax.axvline(0, color='red', lw=1.2, linestyle='--')
        _ax.set_xlabel('Residual (kWh)')
        _ax.set_title(f'{model_name} Residuals  k={HORIZON}s')
        plt.tight_layout()
        plt.savefig(os.path.join(_plots_dir, f'{model_name}_residuals.png'), dpi=180)
        plt.close('all')

    _horizon_results = {}

    # ────────────────────────────────────────────────────────────────────────
    # 1. Linear Regression
    # ────────────────────────────────────────────────────────────────────────
    print('  [1/4] Linear Regression ...')
    _lr = LinearRegression()
    _lr.fit(X_tv_cap, y_tv_cap)
    _pred = _lr.predict(X_te_flat)
    _m = save_benchmark_results(
        'linear_regression', HORIZON, y_test, _pred,
        BENCHMARK_DIR, log_transform=TARGET_LOG_TRANSFORM,
        extra_meta={'training_rows': int(len(X_tv_cap)),
                    'capped_from': int(len(X_tv_flat))}
    )
    _save_pred_plot(y_test, _pred, 'linear_regression')
    joblib.dump(_lr, os.path.join(_models_dir, f'linear_regression_k{HORIZON}.pkl'))
    _horizon_results['linear_regression'] = _m
    print(f'       MAE={_m["MAE"]:.6f}  RMSE={_m["RMSE"]:.6f}  '
          f'R2={_m["R2"]:.4f}  sMAPE={_m["sMAPE"]:.4f}')
    del _lr, _pred;  gc.collect()

    # ────────────────────────────────────────────────────────────────────────
    # 2. SVR  (capped at MAX_SVR_ROWS — SVR is O(n²))
    # ────────────────────────────────────────────────────────────────────────
    print(f'  [2/4] SVR (training on first {MAX_SVR_ROWS:,} rows) ...')
    _svr = SVR(kernel='rbf', C=10, epsilon=0.1)
    _svr.fit(X_tv_flat[:MAX_SVR_ROWS], y_tv[:MAX_SVR_ROWS])
    _pred = _svr.predict(X_te_flat)
    _m = save_benchmark_results(
        'svr', HORIZON, y_test, _pred,
        BENCHMARK_DIR, log_transform=TARGET_LOG_TRANSFORM,
        extra_meta={'training_subset': MAX_SVR_ROWS}
    )
    _save_pred_plot(y_test, _pred, 'svr')
    joblib.dump(_svr, os.path.join(_models_dir, f'svr_k{HORIZON}.pkl'))
    _horizon_results['svr'] = _m
    print(f'       MAE={_m["MAE"]:.6f}  RMSE={_m["RMSE"]:.6f}  '
          f'R2={_m["R2"]:.4f}  sMAPE={_m["sMAPE"]:.4f}')
    del _svr, _pred;  gc.collect()

    # ────────────────────────────────────────────────────────────────────────
    # 3. XGBoost  (uses full splits with early stopping on val)
    # ────────────────────────────────────────────────────────────────────────
    print('  [3/4] XGBoost ...')
    _xgb = XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        early_stopping_rounds=20, eval_metric='rmse',
        random_state=RANDOM_SEED, verbosity=0,
    )
    _xgb.fit(
        X_tr_flat, y_train,
        eval_set=[(X_va_flat, y_val)],
        verbose=False,
    )
    _pred = _xgb.predict(X_te_flat)
    _m = save_benchmark_results(
        'xgboost', HORIZON, y_test, _pred,
        BENCHMARK_DIR, log_transform=TARGET_LOG_TRANSFORM,
        extra_meta={'best_ntree_limit': int(_xgb.best_iteration)}
    )
    _save_pred_plot(y_test, _pred, 'xgboost')
    _xgb.save_model(os.path.join(_models_dir, f'xgboost_k{HORIZON}.json'))
    _horizon_results['xgboost'] = _m
    print(f'       MAE={_m["MAE"]:.6f}  RMSE={_m["RMSE"]:.6f}  '
          f'R2={_m["R2"]:.4f}  sMAPE={_m["sMAPE"]:.4f}')
    del _xgb, _pred;  gc.collect()

    # ────────────────────────────────────────────────────────────────────────
    # 4. MLP (sklearn)
    # ────────────────────────────────────────────────────────────────────────
    print('  [4/4] MLP ...')
    _mlp = MLPRegressor(
        hidden_layer_sizes=(128, 64), activation='relu',
        solver='adam', learning_rate_init=5e-4,
        max_iter=200, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=15,
        random_state=RANDOM_SEED, verbose=False,
    )
    _mlp.fit(X_tv_cap, y_tv_cap)
    _pred = _mlp.predict(X_te_flat)
    _m = save_benchmark_results(
        'mlp', HORIZON, y_test, _pred,
        BENCHMARK_DIR, log_transform=TARGET_LOG_TRANSFORM,
        extra_meta={'training_rows': int(len(X_tv_cap)),
                    'capped_from': int(len(X_tv_flat)),
                    'n_iter': int(_mlp.n_iter_)}
    )
    _save_pred_plot(y_test, _pred, 'mlp')
    joblib.dump(_mlp, os.path.join(_models_dir, f'mlp_k{HORIZON}.pkl'))
    _horizon_results['mlp'] = _m
    print(f'       MAE={_m["MAE"]:.6f}  RMSE={_m["RMSE"]:.6f}  '
          f'R2={_m["R2"]:.4f}  sMAPE={_m["sMAPE"]:.4f}')
    del _mlp, _pred;  gc.collect()

    # ── Per-horizon summary CSV ──────────────────────────────────────────────
    _table = build_comparison_table(BENCHMARK_DIR, HORIZON)
    _csv   = os.path.join(_horizon_dir, 'summary_classical.csv')
    _table.to_csv(_csv, index=False)
    print(f'  Summary CSV → {_csv}')

    # ── Run config JSON for reproducibility ─────────────────────────────────
    with open(os.path.join(_horizon_dir, 'classical_run_config.json'), 'w') as _f:
        json.dump({
            'horizon_k'          : HORIZON,
            'log_transform'      : TARGET_LOG_TRANSFORM,
            'random_seed'        : RANDOM_SEED,
            'max_svr_rows'       : MAX_SVR_ROWS,
            'max_lr_mlp_rows'    : MAX_ML_ROWS,
            'n_test_sequences'   : int(len(y_test)),
            'completed_at'       : datetime.now().isoformat(),
            'metrics'            : _horizon_results,
        }, _f, indent=4)

    all_results[HORIZON] = _horizon_results

    del X_train, y_train, X_val, y_val, X_test, y_test
    del X_tr_flat, X_va_flat, X_te_flat, X_tv_flat, X_tv_cap, y_tv, y_tv_cap
    gc.collect()

_total_h = (time.time() - loop_start) / 3600
print(f'\nAll classical ML benchmarks complete — {_total_h:.2f} h total')


  Horizon 1/4 : k = 60 s   0.00 h elapsed
  Loading canonical splits ...
  Flat train+val : (2276226, 2040)  |  test : (401610, 2040)
  Approx RAM (full TV flat) : 18.57 GB
  LR/MLP capped to 100,000 rows  (from 2,276,226)
  [1/4] Linear Regression ...
       MAE=44.259866  RMSE=135.847937  R2=-0.1187  sMAPE=130.4362
  [2/4] SVR (training on first 50,000 rows) ...
       MAE=44.262049  RMSE=135.846711  R2=-0.1187  sMAPE=153.4979
  [3/4] XGBoost ...
       MAE=44.259796  RMSE=135.847748  R2=-0.1187  sMAPE=112.2957
  [4/4] MLP ...
       MAE=44.682077  RMSE=135.567936  R2=-0.1141  sMAPE=176.7809
  Summary CSV → D:\pipeline5_2\results\benchmark\horizon_60\summary_classical.csv

  Horizon 2/4 : k = 300 s   0.30 h elapsed
  Loading canonical splits ...
  Flat train+val : (2274688, 2040)  |  test : (401338, 2040)
  Approx RAM (full TV flat) : 18.56 GB
  LR/MLP capped to 100,000 rows  (from 2,274,688)
  [1/4] Linear Regression ...
       MAE=64.314131  RMSE=163.767746  R2=-0.1823  sMAPE=85.6

---
## Cross-Horizon Summary

In [5]:
if not all_results:
    print('No results yet — run the main loop first.')
else:
    _rows = []
    for _h, _models in sorted(all_results.items()):
        for _name, _m in _models.items():
            _rows.append({'Horizon_k_s': _h, 'Model': _name, **_m})
    _df = pd.DataFrame(_rows)

    print('Classical ML — all horizons')
    print('=' * 72)
    print(_df.to_string(index=False, float_format='{:.6f}'.format))
    print('=' * 72)

    _csv = os.path.join(ANALYSIS_DIR, 'classical_ml_all_horizons.csv')
    _df.to_csv(_csv, index=False)
    print(f'Saved → {_csv}')

Classical ML — all horizons
 Horizon_k_s             Model       MAE       RMSE        R2      sMAPE
          60 linear_regression 44.259866 135.847937 -0.118746 130.436165
          60               svr 44.262049 135.846711 -0.118726 153.497920
          60           xgboost 44.259796 135.847748 -0.118743 112.295707
          60               mlp 44.682077 135.567936 -0.114139 176.780921
         300 linear_regression 64.314131 163.767746 -0.182309  85.689051
         300               svr 64.313133 163.769056 -0.182328  68.779456
         300           xgboost 64.313897 163.767769 -0.182310  83.236435
         300               mlp 64.335547 163.748891 -0.182037 162.637249
         900 linear_regression 67.149477 167.330667 -0.191830  77.361247
         900               svr 67.146547 167.333463 -0.191870  62.199390
         900           xgboost 67.149499 167.330602 -0.191829  78.014019
         900               mlp 67.164003 167.337208 -0.191924 157.714469
        1800 linear_reg